In [1]:

%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

project_root = Path().resolve().parents[1]
sys.path.append(str(Path().resolve().parents[1]))

In [4]:
import os
import re

import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from io import StringIO
import pandas as pd
import pickle
import igraph as ig
import torch



In [21]:
import torch
from torch.utils.data import DataLoader
from torch.optim import Adam

from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()


from dataset import CachedDataset
from collate import collate_fn

from model import DiffusionGNN
from diff_util import create_diffusion, preprocess


# ----------------------------------
# Config
# ----------------------------------

BATCH_SIZE = 2          # you can increase now
LR = 2e-4
EPOCHS = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NODE_DIM = 3
HIDDEN = 128
TIMESTEPS = 100


# ----------------------------------
# Dataset
# ----------------------------------

dataset = CachedDataset("cached_dataset.pt")
print(f"Dataset size: {len(dataset)}")

loader = DataLoader(

    dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    collate_fn=collate_fn,

    num_workers=0,      # IMPORTANT
    pin_memory=False,    # IMPORTANT

    persistent_workers=False,
)

print(f"Number of batches: {len(loader)}")
model = DiffusionGNN(
    node_dim=NODE_DIM,
    hidden_dim=HIDDEN,
    num_layers=4
).to(DEVICE)

diffusion = create_diffusion(TIMESTEPS)
print(f"Model and diffusion created on {DEVICE}")

optimizer = Adam(model.parameters(), lr=LR)


# ----------------------------------
# Training
# ----------------------------------

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    print(f"Epoch {epoch+1}/{EPOCHS} | Training...")

    for x, adj, node_mask in loader:

        x = x.to(DEVICE, dtype=torch.float16)
        adj = adj.to(DEVICE, dtype=torch.float16) # 500 x 500 adjacency matrix = 250,000 entries = 4000 x 4000 x 4 bytes = 64 MB per graph, batch size 2 = 128 MB just for adj, plus node features and other tensors, so we need to be careful with memory
        node_mask = node_mask.to(DEVICE, dtype=torch.float16)

        # print(adj.shape, node_mask.shape)

        # Mask adj
        adj = adj * node_mask[:, :, None] * node_mask[:, None, :]
        # print(adj.shape, node_mask.shape)
        # print(adj)
        # break
        B = x.shape[0]


        t = torch.randint(
            0,
            diffusion.num_timesteps,
            (B,),
            device=DEVICE
        )


        with autocast():

            loss_dict = diffusion.training_losses(
                model,
                x_start=x,
                t=t,
                model_kwargs={
                    "adj": adj,
                    "node_mask": node_mask
                }
            )

            loss = loss_dict["loss"].mean()


        optimizer.zero_grad()

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()


        total_loss += loss.item()


    avg_loss = total_loss / len(loader)

    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")


    torch.save(model.state_dict(), "model.pt")

Dataset size: 2724
Number of batches: 1362
Model and diffusion created on cuda
Epoch 1/50 | Training...
node_mask in training_losses: tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1.]], device='cuda:0', dtype=torch.float16)


NotImplementedError: 